In [1]:
!pip install scikit-learn torch pandas numpy

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv('./binary_code_awgn_dataset.csv')
num_columns = len(df.columns)
print(f"Number of columns: {num_columns}")

Number of columns: 9217


In [4]:
class ErrorCorrectionDataset:
    def __init__(self, dataframe, snr_db_pools, train_size=0.8, seq_len=4608):
        self.df = dataframe
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.snr_db_pools = snr_db_pools
        self.train_size = train_size
        self.seq_len = seq_len
        self.corrupted_cols = [f"Corrupted_{i}" for i in range(seq_len)]
        self.original_cols = [f"Original_{i}" for i in range(seq_len)]
        self.X = dataframe[self.corrupted_cols].values.astype(np.float32)
        self.Y = dataframe[self.original_cols].values.astype(np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        corrupted = self.X[idx]
        original = self.Y[idx]
        X = torch.tensor(corrupted, dtype=torch.float32).unsqueeze(-1)
        y = torch.tensor(original, dtype=torch.float32)
        return X, y

    def _split_tensor(self, sub_df):
        n_train = int(self.train_size * len(sub_df))
        sub_df = sub_df.sample(frac=1, random_state=None).reset_index(drop=True)
        X = sub_df[self.corrupted_cols].values.astype(np.float32)
        y = sub_df[self.original_cols].values.astype(np.float32)
        X_train, X_test = X[:n_train], X[n_train:]
        y_train, y_test = y[:n_train], y[n_train:]
        return (
            torch.tensor(X_train, dtype=torch.float32).to(self.device),
            torch.tensor(y_train, dtype=torch.float32).to(self.device),
            torch.tensor(X_test, dtype=torch.float32).to(self.device),
            torch.tensor(y_test, dtype=torch.float32).to(self.device),
        )

    def prepare_datasets(self):
        datasets = []
        for snr_db in self.snr_db_pools:
            sub_df = self.df[self.df["SNR_DB"] == snr_db]
            datasets.append(self._split_tensor(sub_df))
        return datasets

    def generalized_dataset(self):
        return self._split_tensor(self.df)

In [5]:
snr_db_pools = [5, 10, 20, 50, 100, 500, 1000]
snr_db_pools.reverse()
Dataset = ErrorCorrectionDataset(df, snr_db_pools, train_size=0.8, seq_len=4608)

error_datasets = Dataset.prepare_datasets()
generalized_dataset = Dataset.generalized_dataset()

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
class AWGNErrorCorrectorBinary(nn.Module):
    def __init__(self, input_dim=4608, d_model=256, nhead=8, num_layers=6, dim_feedforward=1024, dropout=0.1):
        super().__init__()
        self.input_fc = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Multi-layer MLP head
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 1024),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(1024, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, input_dim),
            nn.Sigmoid()  # binary output in [0, 1]
        )

    def forward(self, x):
        # x in range [-1, 1], convert to [0,1] for BCE compatibility
        x = (x + 1) / 2
        x = self.input_fc(x)
        x = self.transformer_encoder(x)
        x = self.mlp(x)
        # Convert output back to [-1,1]
        x = (x * 2) - 1
        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AWGNErrorCorrectorBinary().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def train_on_dataset(X_train, y_train, epochs):
    X_train = X_train.unsqueeze(0)
    y_train = y_train.unsqueeze(0)
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_train)
        # Convert targets to [0,1]
        y_true = (y_train + 1) / 2
        y_pred = (outputs + 1) / 2
        loss = criterion(y_pred, y_true)
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1}, Loss={loss.item():.6f}")

for i, epochs in enumerate([1000, 80, 60, 40, 20, 15, 10]):
    print(f"\n=== Training on SNR={snr_db_pools[i]} for {epochs} epochs ===")
    train_on_dataset(error_datasets[i][0], error_datasets[i][1], epochs)


=== Training on SNR=1000 for 1000 epochs ===
Epoch 1, Loss=0.693918
Epoch 2, Loss=0.693162
Epoch 3, Loss=0.688979
Epoch 4, Loss=0.684636
Epoch 5, Loss=0.682432
Epoch 6, Loss=0.680776
Epoch 7, Loss=0.679413
Epoch 8, Loss=0.679095
Epoch 9, Loss=0.678567
Epoch 10, Loss=0.678580
Epoch 11, Loss=0.678380
Epoch 12, Loss=0.678299
Epoch 13, Loss=0.678160
Epoch 14, Loss=0.677955
Epoch 15, Loss=0.677923
Epoch 16, Loss=0.677807
Epoch 17, Loss=0.677731
Epoch 18, Loss=0.677730
Epoch 19, Loss=0.677659
Epoch 20, Loss=0.677649
Epoch 21, Loss=0.677647
Epoch 22, Loss=0.677601
Epoch 23, Loss=0.677614
Epoch 24, Loss=0.677588
Epoch 25, Loss=0.677554
Epoch 26, Loss=0.677555
Epoch 27, Loss=0.677553
Epoch 28, Loss=0.677525
Epoch 29, Loss=0.677522
Epoch 30, Loss=0.677510
Epoch 31, Loss=0.677486
Epoch 32, Loss=0.677502
Epoch 33, Loss=0.677481
Epoch 34, Loss=0.677484
Epoch 35, Loss=0.677478
Epoch 36, Loss=0.677465
Epoch 37, Loss=0.677460
Epoch 38, Loss=0.677462
Epoch 39, Loss=0.677449
Epoch 40, Loss=0.677457
Epo

In [ ]:
X_train = generalized_dataset[0].unsqueeze(0)
y_train = generalized_dataset[1].unsqueeze(0)
for epoch in range(20):
    optimizer.zero_grad()
    outputs = model(X_train)
    y_true = (y_train + 1) / 2
    y_pred = (outputs + 1) / 2
    loss = criterion(y_pred, y_true)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}, Loss={loss.item():.6f}")

Epoch 1, Loss=0.677391
Epoch 2, Loss=0.677378
Epoch 3, Loss=0.677362
Epoch 4, Loss=0.677352
Epoch 5, Loss=0.677343
Epoch 6, Loss=0.677338
Epoch 7, Loss=0.677339
Epoch 8, Loss=0.677342
Epoch 9, Loss=0.677346
Epoch 10, Loss=0.677347
Epoch 11, Loss=0.677343
Epoch 12, Loss=0.677342
Epoch 13, Loss=0.677340
Epoch 14, Loss=0.677335
Epoch 15, Loss=0.677333
Epoch 16, Loss=0.677336
Epoch 17, Loss=0.677333
Epoch 18, Loss=0.677336
Epoch 19, Loss=0.677335
Epoch 20, Loss=0.677335


In [ ]:
model.eval()
with torch.no_grad():
    X_test = generalized_dataset[2].unsqueeze(0)
    y_test = generalized_dataset[3].unsqueeze(0)
    test_outputs = model(X_test)
    y_true = (y_test + 1) / 2
    y_pred = (test_outputs + 1) / 2
    test_loss = criterion(y_pred, y_true)
    mae = torch.mean(torch.abs(test_outputs - y_test)).item()
    mse = torch.mean((test_outputs - y_test) ** 2).item()
    print(f"Test Loss (BCE): {test_loss.item():.6f}")
    print(f"Mean Abs Error : {mae:.6f}")
    print(f"Mean Sq Error  : {mse:.6f}")

Test Loss (BCE): 0.677396
Mean Abs Error : 0.969032
Mean Sq Error  : 0.968975


In [ ]:
import os
save_path = "./binary_awgn_error_corrector.pth"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")


Model saved to ./binary_awgn_error_corrector.pth


: 

In [ ]:
df = pd.read_csv('/home/network/Documents/Semantic Communications/semacomm-master/error_correction/binary_code_awgn_dataset.csv')
num_columns = len(df.columns)
print(f"Number of columns: {num_columns}")
df.head(10)

Number of columns: 9217


,SNR_DB,Original_0,Original_1,Original_2,Original_3,Original_4,Original_5,Original_6,Original_7,Original_8,...,Corrupted_4598,Corrupted_4599,Corrupted_4600,Corrupted_4601,Corrupted_4602,Corrupted_4603,Corrupted_4604,Corrupted_4605,Corrupted_4606,Corrupted_4607
0,5,1.0,1.0,1.0,-1.0,-1.0,1.0,-1.0,-1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,-1.0,-1.0,-1.0,1.0,1.0
1,10,1.0,1.0,1.0,-1.0,-1.0,1.0,-1.0,-1.0,1.0,...,1.0,1.0,1.0,-1.0,1.0,1.0,-1.0,1.0,1.0,1.0
2,20,1.0,1.0,1.0,-1.0,-1.0,1.0,-1.0,-1.0,1.0,...,1.0,1.0,1.0,1.0,-1.0,1.0,1.0,1.0,1.0,1.0
3,50,1.0,1.0,1.0,-1.0,-1.0,1.0,-1.0,-1.0,1.0,...,1.0,1.0,-1.0,1.0,-1.0,1.0,1.0,1.0,1.0,1.0
4,100,1.0,1.0,1.0,-1.0,-1.0,1.0,-1.0,-1.0,1.0,...,1.0,1.0,1.0,1.0,-1.0,1.0,1.0,1.0,1.0,1.0
5,500,1.0,1.0,1.0,-1.0,-1.0,1.0,-1.0,-1.0,1.0,...,1.0,1.0,1.0,1.0,-1.0,1.0,1.0,1.0,1.0,1.0
6,1000,1.0,1.0,1.0,-1.0,-1.0,1.0,-1.0,-1.0,1.0,...,1.0,1.0,1.0,1.0,-1.0,1.0,1.0,1.0,1.0,1.0
7,5,1.0,1.0,1.0,-1.0,-1.0,1.0,-1.0,-1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
8,10,1.0,1.0,1.0,-1.0,-1.0,1.0,-1.0,-1.0,1.0,...,1.0,1.0,1.0,1.0,-1.0,1.0,1.0,1.0,1.0,1.0
9,20,1.0,1.0,1.0,-1.0,-1.0,1.0,-1.0,-1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [ ]:
row = df.iloc[5]
row

SNR_DB            500.0
Original_0          1.0
Original_1          1.0
Original_2          1.0
Original_3         -1.0
                  ...  
Corrupted_4603      1.0
Corrupted_4604      1.0
Corrupted_4605      1.0
Corrupted_4606      1.0
Corrupted_4607      1.0
Name: 5, Length: 9217, dtype: float64

In [ ]:
row_corrupted = row[[f"Corrupted_{i}" for i in range(4608)]].values.astype(np.float32)
row_corrupted

array([1., 1., 1., ..., 1., 1., 1.], dtype=float32)

In [ ]:
row_original = row[[f"Original_{i}" for i in range(4608)]].values.astype(np.float32)
row_original

array([1., 1., 1., ..., 1., 1., 1.], dtype=float32)

In [ ]:
count = 0
for i in range(4608):
    if row_corrupted[i] != row_original[i]:
        count += 1
        
count

48

In [ ]:
# Load the trained model weights and run inference on `row_corrupted`
weights_path = "/home/network/Documents/Semantic Communications/error_correction/binary_awgn_error_corrector.pth"
device = "cuda"
inference_model = AWGNErrorCorrectorBinary().to(device)
state = torch.load(weights_path, map_location=device)
inference_model.load_state_dict(state)
inference_model.eval()

# Prepare input: shape (1, 1, 4608)
x = torch.tensor(row_corrupted, dtype=torch.float32, device=device).unsqueeze(0).unsqueeze(0)

with torch.no_grad():
    pred = inference_model(x).squeeze(0).squeeze(0)  # shape (4608,)

# Convert to probabilities [0,1] and hard decisions in {-1, 1}
pred_prob = ((pred + 1) / 2).cpu().numpy()
pred_bits = torch.where(pred >= 0, torch.tensor(1.0, device=pred.device), torch.tensor(-1.0, device=pred.device)).cpu().numpy()

# Optional: compare with ground truth if available
y_true = torch.tensor(row_original, dtype=torch.float32, device=device)
bit_acc = (torch.tensor(pred_bits, device=device) == y_true).float().mean().item()

print(f"Loaded weights from: {weights_path}")
print(f"Bit accuracy vs original: {bit_acc:.6f}")
print("Predicted bits (first 32):", pred_bits[:32])

Loaded weights from: /home/network/Documents/Semantic Communications/error_correction/binary_awgn_error_corrector.pth
Bit accuracy vs original: 0.550564
Predicted bits (first 32): [-1.  1.  1.  1. -1.  1.  1. -1.  1. -1.  1. -1.  1. -1.  1.  1.  1. -1.
 -1.  1.  1. -1.  1. -1.  1. -1.  1. -1.  1. -1.  1. -1.]


In [ ]:
row_original[:32]

array([ 1.,  1.,  1., -1., -1.,  1., -1., -1.,  1., -1.,  1.,  1.,  1.,
       -1.,  1., -1., -1.,  1.,  1.,  1.,  1., -1.,  1.,  1.,  1., -1.,
        1., -1.,  1., -1.,  1., -1.], dtype=float32)

In [ ]:
count = 0
for i in range(4608):
    if pred_bits[i] != row_original[i]:
        count += 1
        
count

2071